In [ ]:
pip install findspark

In [ ]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


In [ ]:
import os
import findspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") # Added confirmation message

In [ ]:
import os
import findspark
from pyspark.sql import SparkSession
import pyspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") 

# Define the paths to your local JAR files, now with the file:/// scheme
# Use forward slashes exclusively in the final path string
#mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-11.2.4.jre11.jar"
osql_jar_path = r"file:///C:/spark_drivers/ojdbc17.jar"

spark = SparkSession.builder \
    .appName("SQLServerToosqlTransfer") \
    .config("spark.jars", f"{mssql_jar_path},{osql_jar_path}") \
    .getOrCreate()

print("Spark session created.")

# You can now proceed with your data transfer logic here
# For example:
# df = spark.read.format("jdbc").option(...)

In [ ]:

# SQL Server connection properties
from pyspark.sql import functions as sf
import time

sql_server_url = r"jdbc:sqlserver://HYPE\SQL_2022:1433;databaseName=AdventureSales;encrypt=true;trustServerCertificate=true"
sqlserver_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "fetchSize": "1000"
}
# --- Configuration ---
source_view_name = "sttwk1b" 

# Start the timer
start_time = time.time()

# --- Use the new automatic numeric partitioning parameters ---
partition_column = "year_week_int" 
# Adjust these bounds based on your actual data range (e.g., if you have data from 2022 week 1 to 2025 week 52)
lower_bound = 202201                
upper_bound = 202252                
num_read_partitions = 8          # Set desired parallelism

# 2. Read data from SQL Server using automatic numeric partitioning
df_partitioned = spark.read.format("jdbc") \
    .option("url", sql_server_url) \
    .option("dbtable", source_view_name) \
    .option("partitionColumn", partition_column) \
    .option("lowerBound", lower_bound) \
    .option("upperBound", upper_bound) \
    .option("numPartitions", num_read_partitions) \
    .option("fetchsize", 1000) \
    .option("user", sqlserver_properties["user"]) \
    .option("password", sqlserver_properties["password"]) \
    .option("driver", sqlserver_properties["driver"]) \
    .load() 

# Stop the timer
end_time = time.time()
total_time = end_time - start_time

print(f"DataFrame has {df_partitioned.rdd.getNumPartitions()} partitions.") 
df_partitioned.printSchema() 
print(f"Total time taken for read: {total_time:.2f} seconds")

# Optional: show a few rows
df_partitioned.show(5)


In [ ]:
# Drop the temporary 'yr_wk' column before writing to sqlsvr
df_to_write = df_partitioned.drop("yr_wk")


In [ ]:
# osql connection properties
# Start the timer
start_time = time.time()
# Connection details for osql (Destination)
sqlsvr_host = "localhost"
sqlsvr_port = "1521"
sqlsvr_service_name = "XE" # Standard service name for XE
sqlsvr_user = "system"        
sqlsvr_password = "D@1ry$2014" 
sqlsvr_table = "STTWK"        
sqlsvr_driver_class = "sqlsvr.jdbc.driver.sqlsvrDriver" 

# This is the crucial line: Make sure the URL is formatted exactly like this:
sqlsvr_url = f"jdbc:sqlsvr:thin:@//{sqlsvr_host}:{sqlsvr_port}/{sqlsvr_service_name}"


sqlsvr_jdbc_url =  f"jdbc:sqlsvr:thin:@//{sqlsvr_host}:{sqlsvr_port}/{sqlsvr_service_name}"
 
 
# Write the repartitioned DataFrame to the osql table
df_to_write.write.format("jdbc") \
    .option("url", sqlsvr_url) \
    .option("dbtable", sqlsvr_table) \
    .option("user", sqlsvr_user) \
    .option("password", sqlsvr_password) \
    .option("driver", sqlsvr_driver_class) \
    .option("batchsize", "10000") \
    .mode("append") \
    .save()


print("Data successfully inserted into osql table using 181 parallel processes.")
# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")    

In [ ]:
# Assuming you have successfully run the previous code block which creates `df_partitioned`

from pyspark.sql import functions as sf
import time

# --- Configuration for Destination Table ---
sqlsvr_destination_table = "sttwk_2022a" 

# Define write properties with bulkCopy enabled for maximum speed
write_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "encrypt": "true",
    "trustServerCertificate": "true",
    "bulkCopy": "true" 
}

print(f"Attempting to write to destination table: {sqlsvr_destination_table}")
print(f"DataFrame has {df_partitioned.rdd.getNumPartitions()} partitions for writing.")

# Start timer for write operation
write_start_time = time.time()

try:
    df_partitioned.write \
        .mode("overwrite") \
        .jdbc(
            url=sql_server_url, # Reusing the URL from the read step
            table=sqlsvr_destination_table, 
            properties=write_properties
        )
    
    write_end_time = time.time()
    write_total_time = write_end_time - write_start_time

    print("-" * 50)
    print(f"Data successfully written to table {sqlsvr_destination_table}.")
    print(f"Write operation completed in: {write_total_time:.2f} seconds")
    print("-" * 50)

except Exception as e:
    print(f"An error occurred during the write operation: {e}")


In [ ]:
# Assuming you have successfully run the previous code block which creates `df_partitioned`

import time

# --- Configuration for Destination Table ---
sqlsvr_destination_table = "sttwk_2225" 

# Define write properties with bulkCopy AND truncate enabled
write_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "encrypt": "true",
    "trustServerCertificate": "true",
    "bulkCopy": "true",
    # --- KEY CHANGE: Truncate instead of Drop ---
    "truncate": "true" 
}

print(f"Attempting to truncate and write to destination table: {sqlsvr_destination_table}")
print(f"DataFrame has {df_partitioned.rdd.getNumPartitions()} partitions for writing.")

# Start timer for write operation
write_start_time = time.time()

try:
    df_partitioned.write \
        .mode("overwrite") \
        .jdbc(
            url=sql_server_url, 
            table=sqlsvr_destination_table, 
            properties=write_properties
        )
    
    write_end_time = time.time()
    write_total_time = write_end_time - write_start_time

    print("-" * 50)
    print(f"Data successfully truncated and inserted into table {sqlsvr_destination_table}.")
    print(f"Write operation completed in: {write_total_time:.2f} seconds")
    print("-" * 50)

except Exception as e:
    print(f"An error occurred during the write operation: {e}")



In [ ]:
# Paste the following modified code into your active PySpark shell session
import time
# --- Configuration (UPDATE THESE VALUES) ---
# Start the timer
start_time = time.time()
JDBC_URL = "jdbc:sqlserver://HYPE\\SQL_2022:1433;"
SOURCE_DB_NAME = "AdventureSales" 
#TARGET_TABLE = "Complete_2years_Wk_Report_bySubChannel_All_Dtl_StockPoint"
TARGET_TABLE = "sttwk_2225"
USERNAME = "sa"
PASSWORD = "D@1ry$2014"
DRIVER = "com.microsoft.sqlserver.jdbc.SQLServerDriver" 

# Connection Properties for the READ operation (no change needed here)
read_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
}

# Connection Properties for the WRITE operation
# We rely on the default OVERWRITE behavior (DROP and CREATE TABLE)
write_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
    # Remove "truncate": "true" here
    # --- ADD THIS LINE FOR BULK INSERT SPEED ---
    "bulkCopy": "true",
    # --- KEY CHANGE: Truncate instead of Drop ---
    "truncate": "true" 
}


# --- 1. Define the Source SQL Query (No change) ---
source_sql_query = """
(SELECT 
    *
FROM sttwk1b 
) as custom_query_alias
"""

# --- 2. Load data from SQL Server into a PySpark DataFrame ---
source_df = spark.read.jdbc(
    url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
    table=source_sql_query,
    properties=read_properties
)

print(f"Loaded {source_df.count()} records from source SQL query into PySpark DataFrame.")
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds")  

# --- 3. Overwrite the Target Table (DROP and CREATE TABLE approach) ---
# Start the timer
start_time = time.time()
optimized_df = source_df#.repartition(16)
try:
    # Use the optimized DataFrame
    optimized_df.write \
        .mode("overwrite") \
        .jdbc(
            url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
            table=TARGET_TABLE, 
            properties=write_properties # Make sure you use the bulkCopy properties
        )
    print(f"Successfully wrote data with increased parallelism.")
except Exception as e:
    print(f"An error occurred during write operation: {e}")
    
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds") 

In [ ]:
# Paste the following modified code into your active PySpark shell session
import time
# --- Configuration (UPDATE THESE VALUES) ---
# Start the timer
start_time = time.time()
JDBC_URL = "jdbc:sqlserver://HYPE\\SQL_2022:1433;"
SOURCE_DB_NAME = "AdventureSales" 
#TARGET_TABLE = "Complete_2years_Wk_Report_bySubChannel_All_Dtl_StockPoint"
TARGET_TABLE = "sttwk_2225"
USERNAME = "sa"
PASSWORD = "D@1ry$2014"
DRIVER = "com.microsoft.sqlserver.jdbc.SQLServerDriver" 

# Connection Properties for the READ operation (no change needed here)
read_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
}
# Connection Properties for the WRITE operation
# We rely on the default OVERWRITE behavior (DROP and CREATE TABLE)
write_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
    # Remove "truncate": "true" here
    # --- ADD THIS LINE FOR BULK INSERT SPEED ---
    "bulkCopy": "true",
    # --- KEY CHANGE: Truncate instead of Drop ---
    "truncate": "true" 
}


# --- 1. Define the Source SQL Query (No change) ---
source_sql_query = """
(SELECT 
    *
FROM sttwk1b 
) as custom_query_alias
"""

# --- 2. Load data from SQL Server into a PySpark DataFrame ---
source_df = spark.read.jdbc(
    url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
    table=source_sql_query,
    properties=read_properties
)

print(f"Loaded {source_df.count()} records from source SQL query into PySpark DataFrame.")
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds")  

# --- 3. Overwrite the Target Table (DROP and CREATE TABLE approach) ---
# Start the timer
start_time = time.time()
optimized_df = source_df.repartition(16)
try:
    # Use the optimized DataFrame
    optimized_df.write \
        .mode("overwrite") \
        .jdbc(
            url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
            table=TARGET_TABLE, 
            properties=write_properties # Make sure you use the bulkCopy properties
        )
    print(f"Successfully wrote data with increased parallelism.")
except Exception as e:
    print(f"An error occurred during write operation: {e}")
    
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds")     